# RP-MPQ v2: Importance Mode Comparison

**목표:** NMSE 기반 importance vs rate shortfall의 outage 차이 입증.

## 실험 구성

| Variant | Source | Importance | Binning | 의미 |
|---------|--------|-----------|---------|------|
| Baseline | **기존 결과 재사용** | Rate shortfall (γ=0.99/0.98/0.95) | Zeta | 기존 rpmpq_v2_zeta_sweep.csv |
| D | **새로 실행** | NMSE | Zeta | NMSE만으로 최적화 → outage 열화 예상 |
| E | **새로 실행** | Rate shortfall (γ=0.99) | Hoyer | Hoyer sparsity binning 비교 |

## 재실행 필요 범위
- **collect: 불필요** (perturbation data 재사용)
- **build + eval: D, E만** (GPU, ~15min each)
- **총 예상: ~30min**

## Cell 1: Environment Setup

In [ ]:
import os, sys

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
if os.path.isdir(MAMBAIC_ROOT):
    os.chdir(MAMBAIC_ROOT)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir(MAMBAIC_ROOT)

if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

setup_path = os.path.join(os.path.dirname(MAMBAIC_ROOT), "setup_colab.py")
if os.path.isfile(setup_path):
    exec(open(setup_path).read())
else:
    !pip install -q einops scipy tqdm thop fvcore pybind11

!pip install -q seaborn compressai timm pulp 2>/dev/null | tail -1
print(f"Working dir: {os.getcwd()}")
print("=== Ready ===")

## Cell 2: Verify Data & Load Model

In [ ]:
import importlib
import numpy as np, pandas as pd, torch, argparse, time, shutil
from torch.utils.data import DataLoader

# 모듈 강제 reload (파일 수정 반영)
import rpmpq_v2
importlib.reload(rpmpq_v2)

from rpmpq_v2 import (
    build_importance_surface, evaluate_zeta_sweep, build_kernel,
    get_encoder_block_names, _load_model_and_data,
    RESULTS_CSV, RESULTS_PLOT,
)

# Verify perturbation data + baseline results
required = [
    ("rpmpq_v2_perturbation.csv", "Perturbation data"),
    ("rpmpq_v2_anchor.csv", "Anchor data"),
    ("rpmpq_v2_perfect_rates.csv", "Perfect rates"),
    ("rpmpq_v2_zeta.csv", "Zeta proxy"),
    ("rpmpq_v2_fp32_ref.csv", "FP32 ref + Hoyer"),
    ("rpmpq_v2_zeta_sweep.csv", "Baseline sweep (A)"),
]
for fname, desc in required:
    path = os.path.join(RESULTS_CSV, fname)
    ok = os.path.isfile(path)
    print(f"  [{'OK' if ok else 'MISSING'}] {desc}: {fname}")
    if not ok and 'sweep' not in fname:
        raise FileNotFoundError(f"Run --step collect first: {fname}")

# Load model once
args = argparse.Namespace(
    encoder='mamba', decoder='transnet', encoded_dim=512, M=32,
    encoder_layers=2, decoder_layers=2, use_chunking=False,
    train_path='data/DATA_Htrainout.mat', test_path='data/DATA_Htestout.mat',
    train_key='HT', test_key='HT', no_cuda=False,
    batch_size=256, num_workers=0,
    checkpoint='saved_models/mamba_transnet_L2_dim512_baseline/best.pth',
    aq=8, anchor_bits=16, fc_chunks=32,
)
net, test_loader, norm_params, device = _load_model_and_data(args)
K_d = build_kernel(32, 1.0)
K_a = build_kernel(32, 1.0)
print("\nModel loaded. Ready for comparison experiments.")

## Cell 3: Run D (NMSE importance) + E (Hoyer binning)

Baseline (A)는 기존 `rpmpq_v2_zeta_sweep.csv` 재사용.

**총 예상: ~30min (2 variants × ~15min)**

In [ ]:
SNR_LIST = [10, 20, 30]
TARGET_SAVINGS = np.arange(85, 97.01, 0.25).tolist()
EVAL_GAMMA_LIST = [0.99, 0.98, 0.95]

PERT_CSV = os.path.join(RESULTS_CSV, "rpmpq_v2_perturbation.csv")
ANC_CSV  = os.path.join(RESULTS_CSV, "rpmpq_v2_anchor.csv")
PERF_CSV = os.path.join(RESULTS_CSV, "rpmpq_v2_perfect_rates.csv")
ZETA_CSV = os.path.join(RESULTS_CSV, "rpmpq_v2_zeta.csv")

NEW_VARIANTS = {
    "D_nmse_zeta": {
        "importance_mode": "nmse",
        "binning_var": "zeta",
        "gamma_list": [0.99],
        "label": r"NMSE importance ($\zeta$)",
    },
    "E_shortfall_hoyer_g99": {
        "importance_mode": "shortfall",
        "binning_var": "hoyer",
        "gamma_list": [0.99],
        "label": r"Rate shortfall ($\gamma$=0.99, Hoyer)",
    },
}

for vname, vcfg in NEW_VARIANTS.items():
    print("\n" + "=" * 70)
    print(f"  VARIANT: {vname} — {vcfg['label']}")
    print("=" * 70)
    t0 = time.time()

    prefix = f"rpmpq_v2_{vname}"
    omega_path = os.path.join(RESULTS_CSV, f"{prefix}_omega.npz")

    # BUILD
    print("\n[BUILD] Constructing importance surface...")
    omega_data = build_importance_surface(
        perturbation_csv=PERT_CSV, anchor_csv=ANC_CSV,
        perf_csv=PERF_CSV, zeta_csv=ZETA_CSV,
        model=net, snr_list=SNR_LIST,
        gamma_list=vcfg["gamma_list"],
        importance_mode=vcfg["importance_mode"],
        binning_var=vcfg["binning_var"],
        output_prefix=prefix,
    )

    # EVAL — binning_var 전달하여 lookup 값 매칭
    print("\n[EVAL] Running zeta sweep with ILP...")
    sweep_df = evaluate_zeta_sweep(
        model=net, test_loader=test_loader,
        device=device, norm_params=norm_params,
        snr_list=SNR_LIST, gamma_list=EVAL_GAMMA_LIST,
        omega_path=omega_path,
        K_d=K_d, K_a=K_a, aq_bits=args.aq,
        target_savings=TARGET_SAVINGS,
        binning_var=vcfg["binning_var"],
    )

    # Save with variant name
    src = os.path.join(RESULTS_CSV, "rpmpq_v2_zeta_sweep.csv")
    dst = os.path.join(RESULTS_CSV, f"{prefix}_sweep.csv")
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  Saved: {dst}")

    elapsed = time.time() - t0
    print(f"\n  [{vname}] Done in {elapsed/60:.1f} min")

print("\n" + "=" * 70)
print("  D + E COMPLETE")
print("=" * 70)

## Cell 4: Comparison Plot — NMSE & Outage vs BOPs Saving

Baseline (A) = 기존 `rpmpq_v2_zeta_sweep.csv`

D, E = 방금 실행한 결과

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Image

# All variants: baseline (A) from existing + D, E from new
ALL_VARIANTS = {
    "baseline": {
        "csv": os.path.join(RESULTS_CSV, "rpmpq_v2_zeta_sweep.csv"),
        "label": r"Rate shortfall ($\gamma$=0.99/0.98/0.95, $\zeta$)",
        "ls": "-", "color": "#d62728",
    },
    "D_nmse_zeta": {
        "csv": os.path.join(RESULTS_CSV, "rpmpq_v2_D_nmse_zeta_sweep.csv"),
        "label": r"NMSE importance ($\zeta$)",
        "ls": "--", "color": "#1f77b4",
    },
    "E_shortfall_hoyer": {
        "csv": os.path.join(RESULTS_CSV, "rpmpq_v2_E_shortfall_hoyer_g99_sweep.csv"),
        "label": r"Rate shortfall ($\gamma$=0.99, Hoyer)",
        "ls": "-.", "color": "#9467bd",
    },
}

snr_plot = 20

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: NMSE vs Saving
ax = axes[0]
for vname, vcfg in ALL_VARIANTS.items():
    if not os.path.exists(vcfg["csv"]):
        print(f"  [SKIP] {vname}: {vcfg['csv']} not found")
        continue
    df = pd.read_csv(vcfg["csv"])
    sub = df[df["SNR_Context"] == snr_plot].sort_values("Avg_Saving")
    if len(sub) > 0:
        ax.plot(sub["Avg_Saving"], sub["NMSE_dB"],
                ls=vcfg["ls"], color=vcfg["color"],
                label=vcfg["label"], linewidth=1.5)
ax.set_xlabel("BOPs Saving (%)")
ax.set_ylabel("NMSE (dB)")
ax.set_title(f"NMSE vs Saving (SNR={snr_plot}dB)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Panel 2: Outage γ=0.99
ax = axes[1]
out_col = f"Outage_{snr_plot}dB_99"
for vname, vcfg in ALL_VARIANTS.items():
    if not os.path.exists(vcfg["csv"]):
        continue
    df = pd.read_csv(vcfg["csv"])
    sub = df[df["SNR_Context"] == snr_plot].sort_values("Avg_Saving")
    if len(sub) > 0 and out_col in sub.columns:
        ax.plot(sub["Avg_Saving"], sub[out_col],
                ls=vcfg["ls"], color=vcfg["color"],
                label=vcfg["label"], linewidth=1.5)
ax.set_xlabel("BOPs Saving (%)")
ax.set_ylabel(r"Outage Prob ($\gamma$=0.99)")
ax.set_title(f"Outage vs Saving (SNR={snr_plot}dB)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Panel 3: Outage γ=0.95
ax = axes[2]
out_col = f"Outage_{snr_plot}dB_95"
for vname, vcfg in ALL_VARIANTS.items():
    if not os.path.exists(vcfg["csv"]):
        continue
    df = pd.read_csv(vcfg["csv"])
    sub = df[df["SNR_Context"] == snr_plot].sort_values("Avg_Saving")
    if len(sub) > 0 and out_col in sub.columns:
        ax.plot(sub["Avg_Saving"], sub[out_col],
                ls=vcfg["ls"], color=vcfg["color"],
                label=vcfg["label"], linewidth=1.5)
ax.set_xlabel("BOPs Saving (%)")
ax.set_ylabel(r"Outage Prob ($\gamma$=0.95)")
ax.set_title(f"Outage vs Saving (SNR={snr_plot}dB)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig.suptitle("RP-MPQ v2: Importance Mode Comparison", fontsize=14, y=1.02)
fig.tight_layout()
path = os.path.join(RESULTS_PLOT, "rpmpq_v2_variant_comparison.png")
fig.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {path}")

## Cell 5: Per-Sparsity — Shortfall vs NMSE Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sp_colors = {"High": "#d62728", "Mid": "#ff7f0e", "Low": "#2ca02c"}

compare = [
    (os.path.join(RESULTS_CSV, "rpmpq_v2_zeta_sweep.csv"),
     r"Baseline (shortfall)", "-"),
    (os.path.join(RESULTS_CSV, "rpmpq_v2_D_nmse_zeta_sweep.csv"),
     "NMSE importance", "--"),
]

for si, snr in enumerate([10, 20, 30]):
    ax = axes[si]
    for csv_path, vlabel, vls in compare:
        if not os.path.exists(csv_path):
            continue
        df = pd.read_csv(csv_path)
        sub = df[df["SNR_Context"] == snr].sort_values("Avg_Saving")
        for sp_name in ["High", "Mid", "Low"]:
            col = f"NMSE_dB_{sp_name}"
            if col in sub.columns:
                ax.plot(sub["Avg_Saving"], sub[col],
                        ls=vls, color=sp_colors[sp_name],
                        label=f"{sp_name} ({vlabel})" if si == 0 else "",
                        linewidth=1.3)
    ax.set_title(f"SNR = {snr} dB")
    ax.set_xlabel("BOPs Saving (%)")
    ax.set_ylabel("NMSE (dB)")
    ax.grid(True, alpha=0.3)
    if si == 0:
        ax.legend(fontsize=6, ncol=2)

fig.suptitle("Per-Sparsity: Shortfall vs NMSE Importance", fontsize=14, y=1.02)
fig.tight_layout()
path = os.path.join(RESULTS_PLOT, "rpmpq_v2_sparsity_comparison.png")
fig.savefig(path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {path}")

## Cell 6: Summary Table at ~90% Saving

In [ ]:
target_saving = 90.0
tolerance = 1.0

variant_info = {
    "Baseline (shortfall+zeta)": os.path.join(RESULTS_CSV, "rpmpq_v2_zeta_sweep.csv"),
    "D: NMSE+zeta": os.path.join(RESULTS_CSV, "rpmpq_v2_D_nmse_zeta_sweep.csv"),
    "E: shortfall+Hoyer": os.path.join(RESULTS_CSV, "rpmpq_v2_E_shortfall_hoyer_g99_sweep.csv"),
}

rows = []
for vlabel, csv_path in variant_info.items():
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    for snr in [10, 20, 30]:
        sub = df[df["SNR_Context"] == snr]
        close = sub[(sub["Avg_Saving"] >= target_saving - tolerance) &
                     (sub["Avg_Saving"] <= target_saving + tolerance)]
        if len(close) == 0:
            continue
        best = close.iloc[(close["Avg_Saving"] - target_saving).abs().argsort()[:1]]
        row = {
            "Variant": vlabel, "SNR": snr,
            "Saving": best["Avg_Saving"].values[0],
            "NMSE_dB": best["NMSE_dB"].values[0],
        }
        for g in [99, 98, 95]:
            oc = f"Outage_{snr}dB_{g}"
            if oc in best.columns:
                row[f"Out_{g}"] = best[oc].values[0]
        rows.append(row)

summary = pd.DataFrame(rows)
print(f"\n=== Comparison at ~{target_saving:.0f}% BOPs Saving ===")
display(summary.round(3))

csv_out = os.path.join(RESULTS_CSV, "rpmpq_v2_variant_comparison.csv")
summary.to_csv(csv_out, index=False)
print(f"Saved: {csv_out}")